# Project 9: Collaborative Filtering from Scratch

**Chapter:** Unit 2 — Collaborative Filtering, Recommender Systems

**Goal:** Build a recommender system that predicts missing ratings in a user-item matrix. Implement matrix factorization via alternating minimization and understand the cold-start problem.

**What you'll build:**
1. User-item rating matrix with missing entries
2. Matrix factorization: decompose R ≈ U V^T
3. Alternating minimization (fix U, solve V; fix V, solve U)
4. Predict missing ratings and evaluate

**Rules:** No sklearn (except train/test split). NumPy + matplotlib only.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

## 1. Synthetic Rating Data

We'll generate a user-item matrix with a known low-rank structure, then hide some entries to simulate missing ratings.

In [ ]:
def make_ratings(n_users=50, n_items=30, rank=3, noise=0.3, missing_frac=0.5, seed=42):
    """Generate a low-rank rating matrix with missing entries.
    
    The true matrix R = U_true @ V_true.T + noise
    Then we randomly mask entries as 'missing' (set to 0 / NaN).
    """
    rng = np.random.default_rng(seed)
    
    # True latent factors
    U_true = rng.normal(size=(n_users, rank))
    V_true = rng.normal(size=(n_items, rank))
    R_true = U_true @ V_true.T + rng.normal(scale=noise, size=(n_users, n_items))
    
    # Clip to [1, 5] like real ratings
    R_true = np.clip(R_true, 1, 5)
    
    # Create observation mask (1 = observed, 0 = missing)
    mask = rng.random(size=(n_users, n_items)) > missing_frac
    # Ensure every user and item has at least one rating
    for i in range(n_users):
        if mask[i].sum() == 0:
            mask[i, rng.integers(n_items)] = True
    for j in range(n_items):
        if mask[:, j].sum() == 0:
            mask[rng.integers(n_users), j] = True
    
    R_observed = R_true * mask  # 0 where missing
    
    return R_true, R_observed, mask


R_true, R_obs, mask = make_ratings()
n_users, n_items = R_obs.shape
n_observed = mask.sum()

print(f"Rating matrix: {n_users} users x {n_items} items")
print(f"Observed entries: {n_observed} / {n_users * n_items} ({n_observed / (n_users*n_items):.1%})")
print(f"Rating range: [{R_obs[mask].min():.1f}, {R_obs[mask].max():.1f}]")

# Visualize
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.imshow(R_true, aspect='auto', cmap='YlOrRd', vmin=1, vmax=5)
ax1.set_title('True Ratings (all entries)'); ax1.set_xlabel('Items'); ax1.set_ylabel('Users')
ax2.imshow(np.where(mask, R_obs, np.nan), aspect='auto', cmap='YlOrRd', vmin=1, vmax=5)
ax2.set_title(f'Observed Ratings ({n_observed / (n_users*n_items):.0%} visible)')
ax2.set_xlabel('Items'); ax2.set_ylabel('Users')
plt.tight_layout(); plt.show()

### Train / test split

We split the *observed* ratings into train and test. The model trains on the train set and we evaluate on held-out observed ratings.

In [ ]:
rng = np.random.default_rng(99)
obs_indices = np.argwhere(mask)  # (row, col) pairs
n_obs = len(obs_indices)
perm = rng.permutation(n_obs)
n_train = int(0.8 * n_obs)

train_idx = obs_indices[perm[:n_train]]
test_idx = obs_indices[perm[n_train:]]

train_mask = np.zeros_like(mask)
train_mask[train_idx[:, 0], train_idx[:, 1]] = True
test_mask = np.zeros_like(mask)
test_mask[test_idx[:, 0], test_idx[:, 1]] = True

print(f"Train ratings: {train_mask.sum()}, Test ratings: {test_mask.sum()}")

## 2. Matrix Factorization — Your Turn

### The math

We want to find low-rank matrices $U \in \mathbb{R}^{n \times k}$ and $V \in \mathbb{R}^{m \times k}$ such that:

$$R \approx U V^T$$

**Objective** (over observed entries only):

$$\min_{U, V} \sum_{(i,j) \in \text{observed}} (R_{ij} - \mathbf{u}_i^T \mathbf{v}_j)^2 + \lambda (\|U\|^2 + \|V\|^2)$$

**Alternating minimization:**
1. Fix $V$, solve for each row $\mathbf{u}_i$ using ridge regression
2. Fix $U$, solve for each row $\mathbf{v}_j$ using ridge regression

For user $i$, let $\mathcal{J}_i$ be items they rated. Then:

$$\mathbf{u}_i = (V_{\mathcal{J}_i}^T V_{\mathcal{J}_i} + \lambda I)^{-1} V_{\mathcal{J}_i}^T \mathbf{r}_i$$

where $V_{\mathcal{J}_i}$ is the subset of $V$ rows for items user $i$ rated, and $\mathbf{r}_i$ is the corresponding ratings.

In [ ]:
class MatrixFactorization:
    def __init__(self, n_users, n_items, rank=3, reg=0.1):
        self.rank = rank
        self.reg = reg
        # Initialize randomly
        rng = np.random.default_rng(42)
        self.U = rng.normal(scale=0.1, size=(n_users, rank))
        self.V = rng.normal(scale=0.1, size=(n_items, rank))
        self.loss_history = []
    
    def predict(self, i, j):
        """Predict rating for user i, item j."""
        ### YOUR CODE ###
        # return dot product of U[i] and V[j]
        raise NotImplementedError("Predict: u_i dot v_j")
    
    def predict_all(self):
        """Predict full rating matrix."""
        ### YOUR CODE ###
        # return U @ V.T
        raise NotImplementedError("Full matrix prediction")
    
    def compute_loss(self, R, train_mask):
        """MSE over observed entries + regularization."""
        ### YOUR CODE ###
        # pred = self.predict_all()
        # errors = (R - pred) * train_mask  (zero out unobserved)
        # mse = sum of squared errors / number of observed entries
        # reg = self.reg * (||U||^2 + ||V||^2)
        # return mse + reg
        raise NotImplementedError("Loss = MSE on observed + L2 reg")
    
    def update_user(self, i, R, train_mask):
        """Update user i's latent vector, holding V fixed.
        
        u_i = (V_J^T V_J + lambda * I)^-1  V_J^T  r_i
        where J = items user i has rated.
        """
        ### YOUR CODE ###
        # rated_items = indices where train_mask[i] is True
        # V_sub = self.V[rated_items]           # (|J|, rank)
        # r_sub = R[i, rated_items]              # (|J|,)
        # A = V_sub.T @ V_sub + self.reg * np.eye(self.rank)
        # b = V_sub.T @ r_sub
        # self.U[i] = np.linalg.solve(A, b)
        raise NotImplementedError("Update one user via ridge")
    
    def update_item(self, j, R, train_mask):
        """Update item j's latent vector, holding U fixed.
        
        v_j = (U_I^T U_I + lambda * I)^-1  U_I^T  r_j
        where I = users who rated item j.
        """
        ### YOUR CODE ###
        # Same idea as update_user but transposed
        raise NotImplementedError("Update one item via ridge")
    
    def fit(self, R, train_mask, n_iters=20):
        """Alternating minimization. (Done for you.)"""
        for iteration in range(n_iters):
            # Fix V, update all users
            for i in range(R.shape[0]):
                self.update_user(i, R, train_mask)
            # Fix U, update all items
            for j in range(R.shape[1]):
                self.update_item(j, R, train_mask)
            
            loss = self.compute_loss(R, train_mask)
            self.loss_history.append(loss)
            if iteration % 5 == 0 or iteration == n_iters - 1:
                print(f"  Iter {iteration+1:3d}  |  loss: {loss:.4f}")
        return self

### Sanity checks

In [ ]:
# Small test case
mf_test = MatrixFactorization(n_users=3, n_items=4, rank=2, reg=0.01)
mf_test.U = np.array([[1, 0], [0, 1], [1, 1]], dtype=float)
mf_test.V = np.array([[1, 0], [0, 1], [1, 1], [2, 0]], dtype=float)

# predict test
assert abs(mf_test.predict(0, 0) - 1.0) < 1e-6, "u=[1,0] dot v=[1,0] should be 1"
assert abs(mf_test.predict(2, 2) - 2.0) < 1e-6, "u=[1,1] dot v=[1,1] should be 2"
print("OK: predict correct")

# predict_all test
P = mf_test.predict_all()
assert P.shape == (3, 4), f"Expected (3,4), got {P.shape}"
assert abs(P[0, 3] - 2.0) < 1e-6, "u=[1,0] dot v=[2,0] should be 2"
print("OK: predict_all correct")

# loss should be non-negative
R_small = np.array([[5, 0, 3, 4], [0, 2, 3, 0], [4, 3, 5, 0]], dtype=float)
m_small = (R_small > 0).astype(float)
loss_val = mf_test.compute_loss(R_small, m_small)
assert loss_val >= 0, f"Loss must be non-negative, got {loss_val}"
print(f"OK: Loss = {loss_val:.4f} (non-negative)")

# fit should reduce loss
mf_fit = MatrixFactorization(n_users=3, n_items=4, rank=2, reg=0.01)
loss_before = mf_fit.compute_loss(R_small, m_small)
mf_fit.fit(R_small, m_small, n_iters=10)
loss_after = mf_fit.loss_history[-1]
assert loss_after < loss_before, f"Loss should decrease: {loss_before:.4f} -> {loss_after:.4f}"
print(f"OK: Loss decreased: {loss_before:.4f} -> {loss_after:.4f}")

print("\nAll checks passed!")

## 3. Train & Evaluate

In [ ]:
print("Training matrix factorization (rank=3, lambda=0.1)...")
mf = MatrixFactorization(n_users, n_items, rank=3, reg=0.1)
mf.fit(R_obs, train_mask, n_iters=30)

# Evaluate on test set
R_pred = mf.predict_all()
test_errors = (R_true[test_mask.astype(bool)] - R_pred[test_mask.astype(bool)])
test_rmse = np.sqrt(np.mean(test_errors**2))
print(f"\nTest RMSE: {test_rmse:.3f}")

# Baseline: predict the global mean
global_mean = R_obs[train_mask.astype(bool)].mean()
baseline_errors = R_true[test_mask.astype(bool)] - global_mean
baseline_rmse = np.sqrt(np.mean(baseline_errors**2))
print(f"Baseline (global mean) RMSE: {baseline_rmse:.3f}")
print(f"Improvement: {(1 - test_rmse/baseline_rmse)*100:.1f}%")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(mf.loss_history, 'b-o', markersize=4)
ax1.set_xlabel('Iteration'); ax1.set_ylabel('Loss'); ax1.set_title('Training Loss')
ax1.grid(True, alpha=0.3)

# Compare true vs predicted for a slice of users
sample_users = range(5)
for u in sample_users:
    test_items = np.where(test_mask[u])[0]
    if len(test_items) > 0:
        ax2.scatter(R_true[u, test_items], R_pred[u, test_items], alpha=0.6, s=30)
ax2.plot([1, 5], [1, 5], 'k--', alpha=0.5, label='Perfect')
ax2.set_xlabel('True Rating'); ax2.set_ylabel('Predicted Rating')
ax2.set_title('True vs Predicted (test set)'); ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

## 4. Experiments

### A: Rank selection
Try rank = 1, 2, 3, 5, 10. When does overfitting start?

In [ ]:
ranks = [1, 2, 3, 5, 10]
train_rmses, test_rmses = [], []

for k in ranks:
    model = MatrixFactorization(n_users, n_items, rank=k, reg=0.1)
    model.fit(R_obs, train_mask, n_iters=30)
    pred = model.predict_all()
    
    tr_err = R_obs[train_mask.astype(bool)] - pred[train_mask.astype(bool)]
    te_err = R_true[test_mask.astype(bool)] - pred[test_mask.astype(bool)]
    train_rmses.append(np.sqrt(np.mean(tr_err**2)))
    test_rmses.append(np.sqrt(np.mean(te_err**2)))
    print(f"  rank={k:2d}  train RMSE={train_rmses[-1]:.3f}  test RMSE={test_rmses[-1]:.3f}")

plt.plot(ranks, train_rmses, 'bo-', label='Train RMSE')
plt.plot(ranks, test_rmses, 'rs-', label='Test RMSE')
plt.xlabel('Rank k'); plt.ylabel('RMSE'); plt.title('Effect of Rank on RMSE')
plt.legend(); plt.grid(True, alpha=0.3); plt.show()

**Your observation:** *(At what rank does the test RMSE stop improving? What does that tell you about the data's true dimensionality?)*

...

### B: Regularization
Try lambda = 0.001, 0.01, 0.1, 1.0, 10.0 with rank=3.

In [ ]:
# Your experiment here


## 5. What You Learned

1. Why does matrix factorization work for recommendations? (Think about what the latent factors represent.)
2. Why do we only compute loss on *observed* entries, not the whole matrix?
3. How does regularization prevent overfitting when most entries are missing?
4. What's the cold-start problem? (New user with zero ratings — what does the model predict?)

---
**Next project:** Clustering + EM — unsupervised learning.